In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("../data/email_evaluation_dataset_sinchana.csv")

# Confirmation message
print("Dataset loaded successfully!")

# Display preview
display(df.head())


In [ ]:
df = df.head(100)
df.shape


In [ ]:
df.to_csv("../data/email_evaluation_dataset_sinchana.csv", index=False)


In [ ]:
import re

def clean_email_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)   # remove punctuation & numbers
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    return text


In [ ]:
df['clean_text'] = df['email_text'].apply(clean_email_text)


In [ ]:
print("Clean_text column created successfully!")
display(df[['email_text', 'clean_text']].head())


In [ ]:
def email_assistant(email_text):
    text = email_text.lower()

    # Urgent condition
    if "urgent" in text or "submit" in text or "deadline" in text:
        return "notify", "urgent"

    # thank you emails
    elif "thank you" in text:
        return "ignore", "polite"

    # Default case
    else:
        return "respond", "neutral"


In [ ]:
#Apply email_assistant() to every email
df['assistant_output'] = df['clean_text'].apply(email_assistant)


In [ ]:
# Splits the assistant_output tuples into separate action and tone columns for evaluation

df[['predicted_action', 'predicted_tone']] = pd.DataFrame(
    df['assistant_output'].tolist(),
    index=df.index
)


In [ ]:
df[['predicted_action', 'predicted_tone']].head()


In [ ]:

# Displays expected human-labeled action and tone for comparison with assistant predictions
df[['expected_action', 'expected_tone']].head()



In [ ]:
# Checks whether the assistant's predicted action matches the expected human action and tone matches expected human tone
df['action_correct'] = df['predicted_action'] == df['expected_action']
df['tone_correct'] = df['predicted_tone'] == df['expected_tone']
df.head()

In [ ]:
# Calculates the accuracy of the assistant's action predictions
action_accuracy = df['action_correct'].mean()*100
print("Action Accuracy:", action_accuracy)


In [ ]:
# Calculates the accuracy of the assistant's tone predictions
tone_accuracy = df['tone_correct'].mean()*100
print("Tone Accuracy:", tone_accuracy)


In [ ]:
# Selects emails where the assistant's action prediction is incorrect
error_df = df[df['action_correct'] == False]
error_df[['clean_text', 'expected_action', 'predicted_action']]
error_df = error_df.reset_index(drop=True)
error_df.head()


In [ ]:
# Prints the total number of incorrect action predictions
print("Total incorrect action predictions:", error_df.shape[0])


In [ ]:
# Save the evaluation results to the data folder
df.to_csv("../data/milestone2_output_sinchana.csv", index=False)


## 1. Which type of emails were hardest to classify?
Ambiguous and indirect emails were hardest to classify.
Example: “Please check this when you have time” – sounds polite but may still require action.

## 2. Why did your rules fail in some cases?
The rules depended only on keywords, so emails without those words were misclassified.
Example: “Server CPU usage has crossed 90%” was urgent but lacked words like urgent or deadline.

## 3. How could an LLM improve this process?
An LLM can understand intent and context, not just keywords.
Example: It can correctly identify “Final reminder: internship report submission” as urgent even without explicit keywords

In [ ]:
pip install langsmith


In [ ]:
from langsmith import Client
client = Client()

In [ ]:
#Define the judge prompt
judge_prompt = """You are an evaluator. Compare the model output with the ideal answer.

check:
1.Action correctness
2.Tone correctness

Give score:
1=correct
0=incorrect
"""

In [ ]:
#Run the agent +judge
def evaluate(agent_output,ideal_action,ideal_tone):
    if  agent_output["action"]==ideal_action and agent_output["tone"]==ideal_tone:
        return {"action_score":1,"tone_score":1}
    else:
        return {"action_score":0,"tone_score":0}

In [ ]:
agent_output = {"action":"notify",
                "tone":"urgent"}

In [ ]:
ideal_action = "notify"
ideal_tone = "urgent"

In [ ]:
score = evaluate(agent_output,ideal_action,ideal_tone)
score

In [1]:
import pandas as pd

df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()
print(df.columns)


Index(['id', 'sender', 'subject', 'body', 'priority', 'triage_label',
       'ideal_intent', 'ideal_tone'],
      dtype='object')


In [2]:
def email_assistant(email_text):
    text = email_text.lower()

    if "urgent" in text or "deadline" in text:
        return "notify", "urgent"
    elif "thank you" in text or "thanks" in text:
        return "ignore", "polite"
    else:
        return "respond", "neutral"


In [3]:
predictions = []

for _, row in df.iterrows():
    action, tone = email_assistant(row["body"])
    predictions.append({
        "id": row["id"],
        "predicted_intent": action,
        "predicted_tone": tone
    })

pred_df = pd.DataFrame(predictions)
pred_df.head()


,id,predicted_intent,predicted_tone
0,1,respond,neutral
1,2,respond,neutral
2,3,respond,neutral
3,4,respond,neutral
4,5,respond,neutral


In [4]:
def evaluate(row):
    score = 0
    if row["predicted_intent"] == row["ideal_intent"]:
        score += 1
    if row["predicted_tone"] == row["ideal_tone"]:
        score += 1
    return score

eval_df = df.merge(pred_df, on="id")
eval_df["score"] = eval_df.apply(evaluate, axis=1)

accuracy = (eval_df["score"].sum() / (len(eval_df) * 2)) * 100
accuracy


np.float64(92.0)

In [5]:
eval_df.to_csv("../data/milestone2_output_sinchana.csv", index=False)
